In [1]:
from pathlib import Path
import json
import re
import pandas as pd

def find_ai_lab_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if candidate.name == "ai_lab":
            return candidate
    raise RuntimeError("Không tìm thấy thư mục ai_lab.")

AI_LAB_ROOT = find_ai_lab_root(Path.cwd())

DATASETS_DIR = AI_LAB_ROOT / "datasets"
ARTIFACTS_DIR = AI_LAB_ROOT / "artifacts" / "retriever_v1"
REPORTS_DIR = AI_LAB_ROOT / "reports"

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MEDICAL_KB_PATH = DATASETS_DIR / "medical_kb_v1.json"
KB_CHUNKS_PATH = ARTIFACTS_DIR / "kb_chunks_v1.json"
CHUNK_METADATA_PATH = ARTIFACTS_DIR / "chunk_metadata.json"
CHUNK_STATS_PATH = REPORTS_DIR / "kb_chunk_stats_v1.csv"

print("AI_LAB_ROOT =", AI_LAB_ROOT)
print("MEDICAL_KB_PATH =", MEDICAL_KB_PATH)

AI_LAB_ROOT = D:\homelab\ai_lab
MEDICAL_KB_PATH = D:\homelab\ai_lab\datasets\medical_kb_v1.json


In [2]:
with open(MEDICAL_KB_PATH, "r", encoding="utf-8") as f:
    medical_kb = json.load(f)

print("Số knowledge items:", len(medical_kb))
pd.DataFrame(medical_kb)[["id", "source_id", "title", "section", "risk_level"]]

Số knowledge items: 15


,id,source_id,title,section,risk_level
0,kb_v1_001,nice_sepsis_overview,Khi có dấu hiệu nhiễm trùng và cảm giác rất mệ...,red_flags,high
1,kb_v1_002,nice_sepsis_overview,Một số dấu hiệu bên ngoài làm tăng mức độ cảnh...,red_flags,high
2,kb_v1_003,nice_sepsis_overview,Một số dấu hiệu có thể làm mức độ lo ngại tăng...,red_flags,high
3,kb_v1_004,nice_sepsis_overview,Khi có dấu hiệu nguy cơ cao của nhiễm trùng nặ...,red_flags,high
4,kb_v1_005,blood_tests,Xét nghiệm máu là gì,test_explainers,low
5,kb_v1_006,blood_tests,Vì sao bác sĩ có thể chỉ định xét nghiệm máu,test_explainers,low
6,kb_v1_007,blood_tests,Một số loại xét nghiệm máu thường gặp,test_explainers,low
7,kb_v1_008,blood_tests,Cần chuẩn bị gì trước khi xét nghiệm máu,test_explainers,low
8,kb_v1_009,blood_tests,Sau khi xét nghiệm máu và khi nhận kết quả,test_explainers,low
9,kb_v1_010,chest_pain,Khi nào đau ngực cần gọi cấp cứu ngay,red_flags,high


In [3]:
required_fields = [
    "id", "doc_id", "source_id", "source_name", "source_url",
    "section", "title", "content", "risk_level",
    "tags", "keywords", "test_types",
    "faq_type", "safety_notes", "review_status", "use_in_v1",
    "language", "locale"
]

errors = []

for idx, item in enumerate(medical_kb):
    for field in required_fields:
        if field not in item:
            errors.append({"row": idx, "id": item.get("id"), "missing_field": field})

df_errors = pd.DataFrame(errors)
print("Số lỗi schema:", len(df_errors))
df_errors.head()

Số lỗi schema: 0


""


In [4]:
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def ensure_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        return [x.strip() for x in value.split(",") if x.strip()]
    return []

In [5]:
def build_chunk_text(item):
    parts = [
        f"Tiêu đề: {clean_text(item['title'])}",
        f"Mục: {item['section']}",
        f"Mức độ rủi ro: {item['risk_level']}"
    ]

    keywords = ensure_list(item.get("keywords"))
    tags = ensure_list(item.get("tags"))
    test_types = ensure_list(item.get("test_types"))

    if keywords:
        parts.append(f"Từ khóa: {', '.join(keywords)}")
    if tags:
        parts.append(f"Nhãn: {', '.join(tags)}")
    if test_types:
        parts.append(f"Loại xét nghiệm: {', '.join(test_types)}")
    if item.get("faq_type"):
        parts.append(f"Loại FAQ: {item['faq_type']}")
    if item.get("safety_notes"):
        parts.append(f"Lưu ý an toàn: {clean_text(item['safety_notes'])}")

    parts.append(f"Nội dung: {clean_text(item['content'])}")

    return "\n".join(parts)

In [6]:
kb_chunks = []
chunk_metadata = []

for item in medical_kb:
    chunk = {
        "chunk_id": f"{item['id']}_c1",
        "kb_id": item["id"],
        "doc_id": item["doc_id"],
        "source_id": item["source_id"],
        "source_name": item["source_name"],
        "source_url": item["source_url"],
        "section": item["section"],
        "title": clean_text(item["title"]),
        "content": clean_text(item["content"]),
        "chunk_text": build_chunk_text(item),
        "risk_level": item["risk_level"],
        "tags": ensure_list(item.get("tags")),
        "keywords": ensure_list(item.get("keywords")),
        "test_types": ensure_list(item.get("test_types")),
        "faq_type": item.get("faq_type", ""),
        "safety_notes": clean_text(item.get("safety_notes", "")),
        "review_status": item.get("review_status", "pending"),
        "use_in_v1": bool(item.get("use_in_v1", False)),
        "language": item.get("language", "vi"),
        "locale": item.get("locale", "vi-VN")
    }

    kb_chunks.append(chunk)

    chunk_metadata.append({
        "chunk_id": chunk["chunk_id"],
        "kb_id": chunk["kb_id"],
        "source_id": chunk["source_id"],
        "source_name": chunk["source_name"],
        "section": chunk["section"],
        "title": chunk["title"],
        "risk_level": chunk["risk_level"],
        "faq_type": chunk["faq_type"],
        "use_in_v1": chunk["use_in_v1"]
    })

print("Số chunks tạo ra:", len(kb_chunks))
pd.DataFrame(kb_chunks)[["chunk_id", "title", "section", "risk_level"]]

Số chunks tạo ra: 15


,chunk_id,title,section,risk_level
0,kb_v1_001_c1,Khi có dấu hiệu nhiễm trùng và cảm giác rất mệ...,red_flags,high
1,kb_v1_002_c1,Một số dấu hiệu bên ngoài làm tăng mức độ cảnh...,red_flags,high
2,kb_v1_003_c1,Một số dấu hiệu có thể làm mức độ lo ngại tăng...,red_flags,high
3,kb_v1_004_c1,Khi có dấu hiệu nguy cơ cao của nhiễm trùng nặ...,red_flags,high
4,kb_v1_005_c1,Xét nghiệm máu là gì,test_explainers,low
5,kb_v1_006_c1,Vì sao bác sĩ có thể chỉ định xét nghiệm máu,test_explainers,low
6,kb_v1_007_c1,Một số loại xét nghiệm máu thường gặp,test_explainers,low
7,kb_v1_008_c1,Cần chuẩn bị gì trước khi xét nghiệm máu,test_explainers,low
8,kb_v1_009_c1,Sau khi xét nghiệm máu và khi nhận kết quả,test_explainers,low
9,kb_v1_010_c1,Khi nào đau ngực cần gọi cấp cứu ngay,red_flags,high


In [7]:
stats_rows = []

for chunk in kb_chunks:
    stats_rows.append({
        "chunk_id": chunk["chunk_id"],
        "source_id": chunk["source_id"],
        "section": chunk["section"],
        "risk_level": chunk["risk_level"],
        "title_len": len(chunk["title"]),
        "content_len": len(chunk["content"]),
        "chunk_text_len": len(chunk["chunk_text"]),
        "keyword_count": len(chunk["keywords"]),
        "tag_count": len(chunk["tags"])
    })

df_stats = pd.DataFrame(stats_rows)
df_stats.to_csv(CHUNK_STATS_PATH, index=False, encoding="utf-8-sig")

print("Đã ghi:", CHUNK_STATS_PATH)
df_stats.describe(include="all")

Đã ghi: D:\homelab\ai_lab\reports\kb_chunk_stats_v1.csv


,chunk_id,source_id,section,risk_level,title_len,content_len,chunk_text_len,keyword_count,tag_count
count,15,15,15,15,15.000000,15.000000,15.000000,15.000000,15.000000
unique,15,4,2,2,NaN,NaN,NaN,NaN,NaN
top,kb_v1_001_c1,blood_tests,red_flags,high,NaN,NaN,NaN,NaN,NaN
freq,1,5,10,10,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,49.733333,220.866667,506.666667,2.866667,2.266667
std,NaN,NaN,NaN,NaN,20.197124,38.412919,69.623135,0.990430,0.457738
min,NaN,NaN,NaN,NaN,20.000000,148.000000,372.000000,2.000000,2.000000
25%,NaN,NaN,NaN,NaN,37.000000,193.500000,470.500000,2.000000,2.000000
50%,NaN,NaN,NaN,NaN,42.000000,236.000000,509.000000,3.000000,2.000000
75%,NaN,NaN,NaN,NaN,65.000000,242.000000,543.500000,3.500000,2.500000


In [8]:
with open(KB_CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(kb_chunks, f, ensure_ascii=False, indent=2)

with open(CHUNK_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(chunk_metadata, f, ensure_ascii=False, indent=2)

print("Đã ghi:", KB_CHUNKS_PATH)
print("Đã ghi:", CHUNK_METADATA_PATH)

Đã ghi: D:\homelab\ai_lab\artifacts\retriever_v1\kb_chunks_v1.json
Đã ghi: D:\homelab\ai_lab\artifacts\retriever_v1\chunk_metadata.json


In [9]:
for chunk in kb_chunks[:3]:
    print("=" * 100)
    print("CHUNK ID:", chunk["chunk_id"])
    print("TITLE   :", chunk["title"])
    print("SECTION :", chunk["section"])
    print("TEXT:\n")
    print(chunk["chunk_text"][:2000])
    print("\n")

CHUNK ID: kb_v1_001_c1
TITLE   : Khi có dấu hiệu nhiễm trùng và cảm giác rất mệt hoặc rất không ổn, cần được đánh giá sớm
SECTION : red_flags
TEXT:

Tiêu đề: Khi có dấu hiệu nhiễm trùng và cảm giác rất mệt hoặc rất không ổn, cần được đánh giá sớm
Mục: red_flags
Mức độ rủi ro: high
Từ khóa: nhiễm trùng nặng, sepsis, dấu hiệu nặng
Nhãn: sepsis, red_flags, infection
Loại FAQ: red_flag_general
Lưu ý an toàn: Không dùng nội dung này để tự kết luận sepsis.
Nội dung: Nếu một người có dấu hiệu nhiễm trùng và đồng thời cảm thấy rất mệt, rất yếu hoặc diễn tiến xấu nhanh, cần nghĩ đến khả năng bệnh nặng hơn bình thường và nên được đánh giá y tế sớm.


CHUNK ID: kb_v1_002_c1
TITLE   : Một số dấu hiệu bên ngoài làm tăng mức độ cảnh giác nhiễm trùng nặng
SECTION : red_flags
TEXT:

Tiêu đề: Một số dấu hiệu bên ngoài làm tăng mức độ cảnh giác nhiễm trùng nặng
Mục: red_flags
Mức độ rủi ro: high
Từ khóa: da tái xám, tím môi, ban không mất màu, nhiễm trùng nặng
Nhãn: sepsis, red_flags, skin_signs
Loại FA